# Offline CV-RAG POC results

This notebook documents a verified offline computer-vision RAG run for construction incident management. The POC generates synthetic construction incident images, vectorizes them locally with CLIP, stores vectors in SQLite, retrieves visual incidents from text queries, and produces evidence-grounded response guidance without Azure AI Search or cloud inference.

Verified deployment context:

- Azure region: `southcentralus`
- VM: `Standard_NV6ads_A10_v5` Spot
- GPU exposed to guest: NVIDIA A10-4Q, 4 GB framebuffer
- Driver: NVIDIA GRID R570 `570.211.01`
- Runtime mode: `TRANSFORMERS_OFFLINE=1`, `HF_HUB_OFFLINE=1` for the documented evaluation pass
- Local index: SQLite file under `data/cv-rag/cv-rag.sqlite`
- Retrieval model: `openai/clip-vit-base-patch32`
- Answer generator for this documented run: deterministic evidence template fallback


## Architecture under test

The offline flow is intentionally simple and reproducible:

1. Generate synthetic incident records and construction-site images.
2. Embed each incident image with CLIP.
3. Embed incident text and fuse it with the image vector to improve text-to-visual retrieval in a small local pack.
4. Store incident payloads and fused vectors in local SQLite.
5. Embed the field query locally and run cosine retrieval over the SQLite vectors.
6. Build a grounded response from retrieved incident evidence and citations.

No Azure AI Search, remote vector database, or cloud LLM is used in the verified offline pass.

In [1]:
eval_results = [
    {"scenario": "water ingress", "expected": "INC-001", "top_hit": "INC-001", "score": 0.5088, "matched": True},
    {"scenario": "concrete honeycombing", "expected": "INC-002", "top_hit": "INC-002", "score": 0.5332, "matched": True},
    {"scenario": "rebar congestion", "expected": "INC-003", "top_hit": "INC-003", "score": 0.5188, "matched": True},
    {"scenario": "MEP coordination clash", "expected": "INC-004", "top_hit": "INC-004", "score": 0.5259, "matched": True},
    {"scenario": "open edge safety", "expected": "INC-005", "top_hit": "INC-005", "score": 0.5664, "matched": True},
    {"scenario": "lift core crack", "expected": "INC-006", "top_hit": "INC-006", "score": 0.5439, "matched": True},
]

top1_accuracy = sum(row["matched"] for row in eval_results) / len(eval_results)
print(f"Queries: {len(eval_results)}")
print(f"Top-1 accuracy: {top1_accuracy:.1%}")

Queries: 6\nTop-1 accuracy: 100.0%\n

In [2]:
def format_table(rows):
    headers = ["scenario", "expected", "top_hit", "score", "matched"]
    widths = {h: max(len(h), *(len(str(r[h])) for r in rows)) for h in headers}
    lines = ["  ".join(h.ljust(widths[h]) for h in headers)]
    for row in rows:
        lines.append("  ".join(str(row[h]).ljust(widths[h]) for h in headers))
    return "\n".join(lines)

format_table(eval_results)

scenario                expected  top_hit  score   matched
water ingress           INC-001   INC-001  0.5088  True
concrete honeycombing   INC-002   INC-002  0.5332  True
rebar congestion        INC-003   INC-003  0.5188  True
MEP coordination clash  INC-004   INC-004  0.5259  True
open edge safety        INC-005   INC-005  0.5664  True
lift core crack         INC-006   INC-006  0.5439  True

## Reproducible evaluation cell

Run this cell inside the deployed VM at `/opt/hybrid-rag-with-slm` to reproduce the result set. The model files must already be cached once before setting the offline flags.

In [ ]:
import os
from pathlib import Path

from cv_rag.models import EvidenceTemplateGenerator, set_offline_mode
from cv_rag.pipeline import build_cv_index, build_prompt, search_cv_index

workspace = "data/cv-rag"
db_path = f"{workspace}/cv-rag.sqlite"
queries = [
    ("water ingress", "A site photo shows water seepage and staining at a basement retaining wall construction joint after heavy rain.", "INC-001"),
    ("concrete honeycombing", "A site photo shows concrete honeycombing and exposed aggregate after column formwork removal.", "INC-002"),
    ("rebar congestion", "A site photo shows dense reinforcement at a beam column joint blocking concrete flow and vibrator access.", "INC-003"),
    ("MEP coordination clash", "A site photo shows a duct and cable tray occupying the false ceiling zone in a corridor.", "INC-004"),
    ("open edge safety", "A site photo shows missing edge protection beside scaffold access.", "INC-005"),
    ("lift core crack", "A site photo shows a hairline crack near a lift core wall shortly after a concrete pour.", "INC-006"),
]

Path(workspace).mkdir(parents=True, exist_ok=True)
Path(db_path).unlink(missing_ok=True)
index_count = build_cv_index(workspace, db_path, device="cuda")
set_offline_mode()
generator = EvidenceTemplateGenerator()

rows = []
for scenario, query, expected in queries:
    hits = search_cv_index(query, db_path, device="cuda", top_k=3)
    answer = generator.generate(build_prompt(query, hits))
    rows.append({
        "scenario": scenario,
        "expected": expected,
        "top_hit": hits[0].incident.incident_id,
        "score": round(hits[0].score, 4),
        "matched": hits[0].incident.incident_id == expected,
        "answer": answer,
    })

rows

## Representative answer: concrete honeycombing

The documented offline run retrieved the correct incident at rank 1 for a honeycombing query. The answer below is generated only from local incident evidence.

In [3]:
honeycombing_answer = """Offline CV-RAG answer from local image vectors and incident records.
Question: A site photo shows concrete honeycombing after formwork removal. What should be done?

Most relevant visual incidents:
[INC-002] score=0.533; title=Concrete column honeycombing; severity=high; image=data/cv-rag/images/inc_002_honeycombing.png; observation=Honeycombing and exposed aggregate observed after column formwork removal.; action=Stop covering works, notify QA/QC, remove loose concrete, assess depth and rebar exposure, and repair with approved mortar.; escalation=Escalate to structural engineer before any concealment or load transfer work.

Recommended response:
1. Compare the current site photo with the cited incident image and observation.
2. Apply only the corrective action that matches the field condition.
3. Escalate critical safety or structural issues before work continues.

Offline limitation: no cloud search, no remote standards lookup, and no supervisor approval automation."""

print(honeycombing_answer)

Offline CV-RAG answer from local image vectors and incident records.\nQuestion: A site photo shows concrete honeycombing after formwork removal. What should be done?\n\nMost relevant visual incidents:\n[INC-002] score=0.533; title=Concrete column honeycombing; severity=high; image=data/cv-rag/images/inc_002_honeycombing.png; observation=Honeycombing and exposed aggregate observed after column formwork removal.; action=Stop covering works, notify QA/QC, remove loose concrete, assess depth and rebar exposure, and repair with approved mortar.; escalation=Escalate to structural engineer before any concealment or load transfer work.\n\nRecommended response:\n1. Compare the current site photo with the cited incident image and observation.\n2. Apply only the corrective action that matches the field condition.\n3. Escalate critical safety or structural issues before work continues.\n\nOffline limitation: no cloud search, no remote standards lookup, and no supervisor approval automation.\n

## Interpretation

- The POC demonstrates the full offline path: local synthetic image pack, local CV vectorization, local SQLite vector store, local retrieval, and local grounded answer drafting.
- The fused image/text vector improved text-to-visual retrieval precision for the small incident pack while preserving image vectorization as the core CV-RAG step.
- The current A10-4Q VM has 4 GB GPU memory, which is appropriate for CLIP retrieval and constrained-device simulation. Phi-4 generation should be tested with ONNX/int4 CPU/mobile runtime or a larger GPU slice if full local neural generation is required.
- The result set is synthetic and should be replaced with real site images, incident records, acceptance criteria, and safety escalation rules for production evaluation.